<a href="https://colab.research.google.com/github/tinana2k/CS-5530---Tina-Nguyen/blob/main/Assignments/Assignment%202%20%26%203/Q1_Used%20Cars/src/Q1_Used_Cars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1 - Used Cars Data Cleaning and Preprocessing

This notebook completes **Q1** of the assignment using the used-cars dataset.

## Tasks covered
- (a) handling missing values
- (b) removing units from attributes
- (c) one-hot encoding categorical variables
- (d) creating new features
- (e) performing select, filter, rename, mutate, arrange, and summarize/groupby operations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

## **A) Look for the missing values in all the columns and either impute them (replace with mean, median, or mode) or drop them. Justify your action for this task.**



In [17]:
# --- Import Libraries ---
import pandas as pd

# --- Load Dataset from your GitHub ---
url = "https://raw.githubusercontent.com/tinana2k/CS-5530---Tina-Nguyen/main/Assignments/Assignment%202%20%26%203/Q1_Used%20Cars/data_raw/train.csv"

used_cars = pd.read_csv(url)

print("✅ Original Data Loaded")
print("📊 Shape:", used_cars.shape)

# Equivalent of glimpse()
used_cars.info()

# Show first rows
used_cars.head()

✅ Original Data Loaded
📊 Shape: (5847, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   object 
 9   Engine             5811 non-null   object 
 10  Power              5811 non-null   object 
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    object 
 13  Price              5847 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 639.6+ KB


,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
2,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
3,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74
4,6,Nissan Micra Diesel XV,Jaipur,2013,86999,Diesel,Manual,First,23.08 kmpl,1461 CC,63.1 bhp,5.0,NaN,3.50


In [22]:
# Check to see NAs or missing values
print("--- Summary of original Data ---")
print(used_cars.describe(include="all"))

for col_name in used_cars.columns:
    na_count = used_cars[col_name].isna().sum()
    if na_count > 0:
        print(f"Column '{col_name}' has {na_count} NA values.")
    else:
        print(f"Column '{col_name}' has no NA values.")

# Check unique values for certain columns
columns_to_check = ["Mileage", "Engine", "Power", "New_Price"]
excluded_columns = ["X", "Name"] + columns_to_check

print("\n--- Unique values for Character (String) Columns ---")

for col_name in used_cars.columns:
    if used_cars[col_name].dtype == "object" and col_name not in excluded_columns:
        print(f"\nUnique values for '{col_name}':")
        print(used_cars[col_name].dropna().unique())

for col_name in columns_to_check:
    print(f"\nUnique last 4 characters for '{col_name}':")
    print(used_cars[col_name].dropna().astype(str).str[-4:].unique())

# Check the column 'Year' in valid range 1975 - 2024
min_allowed_year = 1975
max_allowed_year = 2024

invalid_years = used_cars[
    (used_cars["Year"] < min_allowed_year) | (used_cars["Year"] > max_allowed_year)
]

if len(invalid_years) > 0:
    print("Found invalid years:")
    print(invalid_years[["Name", "Year"]].head())
else:
    print("All years are valid.")

# Identify potentially invalid Kilometers_Driven values
problematic_km_driven = used_cars[used_cars["Kilometers_Driven"] > 1000000]

if len(problematic_km_driven) > 0:
    print(f"Found {len(problematic_km_driven)} rows with 'Kilometers_Driven' > 1,000,000 km.")

    # Replace likely data entry error
    used_cars["Kilometers_Driven"] = used_cars["Kilometers_Driven"].replace(6500000, 650000)
else:
    print("No extreme 'Kilometers_Driven' outliers (above 1,000,000 km) found to replace.")

# Check rows with problematic units
problematic_units_rows = used_cars[
    (~used_cars["Mileage"].astype(str).str.contains("kmpl", na=False)) |
    (~used_cars["Engine"].astype(str).str.contains("CC", na=False)) |
    (~used_cars["Power"].astype(str).str.contains("bhp", na=False))
]

problematic_units_rows

--- Summary of original Data ---
         Unnamed: 0                    Name Location         Year  \
count   5847.000000                    5847     5847  5847.000000   
unique          NaN                    1804       11          NaN   
top             NaN  Mahindra XUV500 W8 2WD   Mumbai          NaN   
freq            NaN                      49      762          NaN   
mean    3013.181461                     NaN      NaN  2013.448435   
std     1736.398890                     NaN      NaN     3.194949   
min        1.000000                     NaN      NaN  1998.000000   
25%     1509.500000                     NaN      NaN  2012.000000   
50%     3015.000000                     NaN      NaN  2014.000000   
75%     4517.500000                     NaN      NaN  2016.000000   
max     6018.000000                     NaN      NaN  2019.000000   

        Kilometers_Driven Fuel_Type Transmission Owner_Type    Mileage  \
count        5.847000e+03      5847         5847       5847     

,Unnamed: 0,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
1,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,13 km/kg,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
24,26,Nissan Micra Diesel XV,Hyderabad,2012,54000,Diesel,Manual,First,16.48 km/kg,1461 CC,63.1 bhp,5.0,NaN,4.25
56,58,Nissan X-Trail SLX AT,Hyderabad,2010,121812,Diesel,Automatic,First,9.49 km/kg,1995 CC,147.6 bhp,5.0,NaN,7.75
186,194,Honda City 1.5 GXI,Ahmedabad,2007,60006,Petrol,Manual,First,0.0 kmpl,NaN,NaN,NaN,NaN,2.95
200,208,Maruti Swift 1.3 VXi,Kolkata,2010,42001,Petrol,Manual,First,16.1 kmpl,NaN,NaN,NaN,NaN,2.11
709,733,Maruti Swift 1.3 VXi,Chennai,2006,97800,Petrol,Manual,Third,16.1 kmpl,NaN,NaN,NaN,NaN,1.75
723,749,Land Rover Range Rover 3.0 D,Mumbai,2008,55001,Diesel,Automatic,Second,0.0 kmpl,NaN,NaN,NaN,NaN,26.50
1253,1294,Honda City 1.3 DX,Delhi,2009,55005,Petrol,Manual,First,12.8 kmpl,NaN,NaN,NaN,NaN,3.20
1284,1327,Maruti Swift 1.3 ZXI,Hyderabad,2015,50295,Petrol,Manual,First,16.1 kmpl,NaN,NaN,NaN,NaN,5.80
1339,1385,Honda City 1.5 GXI,Pune,2004,115000,Petrol,Manual,Second,0.0 kmpl,NaN,NaN,NaN,NaN,1.50


## **b) Remove the units from some of the attributes and only keep the numerical values (for example remove kmpl from “Mileage”, CC from “Engine”, bhp from “Power”, and lakh from “New_price”).**

In [26]:
import pandas as pd
import numpy as np

# --- b. Remove units and keep only numerical values ---

cng_to_petrol_equivalent = 1.6
cr_to_lakh = 100

used_cars_cleaned = used_cars.copy()

# -------------------
# Mileage
# -------------------
mileage_str = used_cars_cleaned["Mileage"].astype(str)

# extract the number only
mileage_num = pd.to_numeric(
    mileage_str.str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

used_cars_cleaned["Mileage"] = np.where(
    mileage_str.str.contains("km/kg", na=False),
    mileage_num * cng_to_petrol_equivalent,
    np.where(
        mileage_str.str.contains("kmpl", na=False),
        mileage_num,
        np.nan
    )
)

# -------------------
# Engine
# -------------------
used_cars_cleaned["Engine"] = pd.to_numeric(
    used_cars_cleaned["Engine"].astype(str).str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

# -------------------
# Power
# -------------------
used_cars_cleaned["Power"] = pd.to_numeric(
    used_cars_cleaned["Power"].astype(str).str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

# -------------------
# New_Price
# -------------------
new_price_str = used_cars_cleaned["New_Price"].astype(str)

new_price_num = pd.to_numeric(
    new_price_str.str.extract(r"([\d.]+)")[0],
    errors="coerce"
)

used_cars_cleaned["New_Price"] = np.where(
    new_price_str.str.contains("cr", case=False, na=False),
    new_price_num * cr_to_lakh,
    np.where(
        new_price_str.str.contains("Lakh", case=False, na=False),
        new_price_num,
        np.nan
    )
)

print("Cleaned data info:")
used_cars_cleaned.info()

print("\nSummary:")
print(used_cars_cleaned.describe(include="all"))

Cleaned data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   float64
 9   Engine             5811 non-null   float64
 10  Power              5811 non-null   float64
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    float64
 13  Price              5847 non-null   float64
dtypes: float64(6), int64(3), object(5)
memory usage: 639.6+ KB

Summary:
         Unnamed: 0             

## **C) Change the categorical variables (“Fuel_Type” and “Transmission”) into numerical one hot encoded value.**

In [39]:
# --- c. Transform Categorical Variables into One-Hot Encoded Values ---

import pandas as pd
from pathlib import Path

# Make a copy from previous step (after feature engineering)
encoded_used_cars = engineer_used_cars.copy()

# Apply One-Hot Encoding (same as dummy_cols in R)
encoded_used_cars = pd.get_dummies(
    encoded_used_cars,
    columns=["Fuel_Type", "Transmission", "Brand", "Location"],
    drop_first=False
)

# Convert boolean columns to 1/0 (just in case)
bool_cols = encoded_used_cars.select_dtypes(include="bool").columns
encoded_used_cars[bool_cols] = encoded_used_cars[bool_cols].astype(int)

# --- Display Output ---
print("--- Data After One-Hot Encoding (Task c) ---")

print("\nFirst 6 rows:")
display(encoded_used_cars.head(6))

print("\nData Info (similar to glimpse):")
encoded_used_cars.info()

print("\nTotal Rows:", encoded_used_cars.shape[0])
print("Total Columns:", encoded_used_cars.shape[1])

# --- Save to CSV (FIXED PATH ISSUE) ---
save_dir = Path("../data_clean")
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "clean_used_cars.csv"
encoded_used_cars.to_csv(save_path, index=False)

print("\n✅ Dataframe successfully saved to:", save_path.resolve())

--- Data After One-Hot Encoding (Task c) ---

First 6 rows:


,Unnamed: 0,Name,Year,Kilometers_Driven,Owner_Type,Mileage,Engine,Power,Seats,New_Price,...,Location_Bangalore,Location_Chennai,Location_Coimbatore,Location_Delhi,Location_Hyderabad,Location_Jaipur,Location_Kochi,Location_Kolkata,Location_Mumbai,Location_Pune
0,1,Hyundai Creta 1.6 CRDi SX Option,2015,41000,First,19.67,1582.0,126.20,5.0,12.500,...,0,0,0,0,0,0,0,0,0,1
1,2,Honda Jazz V,2011,46000,First,20.80,1199.0,88.70,5.0,8.610,...,0,1,0,0,0,0,0,0,0,0
2,3,Maruti Ertiga VDI,2012,87000,First,20.77,1248.0,88.76,7.0,11.215,...,0,1,0,0,0,0,0,0,0,0
3,4,Audi A4 New 2.0 TDI Multitronic,2013,40670,Second,15.20,1968.0,140.80,5.0,17.740,...,0,0,1,0,0,0,0,0,0,0
4,6,Nissan Micra Diesel XV,2013,86999,First,23.08,1461.0,63.10,5.0,11.750,...,0,0,0,0,0,1,0,0,0,0
5,7,Toyota Innova Crysta 2.8 GX AT 8S,2016,36000,First,11.36,2755.0,171.50,8.0,21.000,...,0,0,0,0,0,0,0,0,1,0



Data Info (similar to glimpse):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 60 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              5847 non-null   int64  
 1   Name                    5847 non-null   object 
 2   Year                    5847 non-null   int64  
 3   Kilometers_Driven       5847 non-null   int64  
 4   Owner_Type              5847 non-null   object 
 5   Mileage                 5847 non-null   float64
 6   Engine                  5847 non-null   float64
 7   Power                   5847 non-null   float64
 8   Seats                   5847 non-null   float64
 9   New_Price               5847 non-null   float64
 10  Price                   5847 non-null   float64
 11  Car_Age                 5847 non-null   int64  
 12  Kilometers_Per_Year     5847 non-null   float64
 13  Is_First_Owner          5847 non-null   int64  
 14  Fuel_Ty

In [40]:
print("Total columns:", encoded_used_cars.shape[1])

print("\nFuel types:")
print(encoded_used_cars.filter(like="Fuel_Type_").columns)

print("\nTransmission types:")
print(encoded_used_cars.filter(like="Transmission_").columns)

print("\nBrand count:", len(encoded_used_cars.filter(like="Brand_").columns))
print("Location count:", len(encoded_used_cars.filter(like="Location_").columns))

Total columns: 60

Fuel types:
Index(['Fuel_Type_Diesel', 'Fuel_Type_Electric', 'Fuel_Type_Petrol'], dtype='object')

Transmission types:
Index(['Transmission_Automatic', 'Transmission_Manual'], dtype='object')

Brand count: 30
Location count: 11


In [41]:
["Fuel_Type_", "Transmission_", "Brand_", "Location_"]

['Fuel_Type_', 'Transmission_', 'Brand_', 'Location_']

## **d) Create one more feature and add this column to the dataset (you can use mutate function in R for this). For example, you can calculate the current age of the car by subtracting “Year” value from the current year.**

In [33]:
from datetime import datetime

# Make a copy
engineer_used_cars = imputed_used_cars.copy()

# Current year
current_year = datetime.now().year

# Task d: create new features
engineer_used_cars["Car_Age"] = current_year - engineer_used_cars["Year"]
engineer_used_cars["Brand"] = engineer_used_cars["Name"].astype(str).str.split().str[0]
engineer_used_cars["Kilometers_Per_Year"] = (
    engineer_used_cars["Kilometers_Driven"] / (engineer_used_cars["Car_Age"] + 1)
).round(0)
engineer_used_cars["Is_First_Owner"] = (
    engineer_used_cars["Owner_Type"] == "First"
).astype(int)

print("--- Data with New Engineered Features (Task d) ---")

engineer_used_cars[
    ["Name", "Brand", "Year", "Car_Age", "Kilometers_Per_Year", "Owner_Type", "Is_First_Owner"]
].head(6)

--- Data with New Engineered Features (Task d) ---


,Name,Brand,Year,Car_Age,Kilometers_Per_Year,Owner_Type,Is_First_Owner
0,Hyundai Creta 1.6 CRDi SX Option,Hyundai,2015,11,3417.0,First,1
1,Honda Jazz V,Honda,2011,15,2875.0,First,1
2,Maruti Ertiga VDI,Maruti,2012,14,5800.0,First,1
3,Audi A4 New 2.0 TDI Multitronic,Audi,2013,13,2905.0,Second,0
4,Nissan Micra Diesel XV,Nissan,2013,13,6214.0,First,1
5,Toyota Innova Crysta 2.8 GX AT 8S,Toyota,2016,10,3273.0,First,1


## **e) Perform select, filter, rename, mutate, arrange and summarize with group by operations (or their equivalent operations in python) on this dataset.**

In [46]:
# --- e. Perform select, filter, rename, mutate, arrange and summarize with group by ---

import pandas as pd
from pathlib import Path

print("--- Demonstrating pandas operations (Task e) ---")

# Use engineer_used_cars because it still has Brand and Location columns
analysis_goal = engineer_used_cars[
    ["Brand", "Location", "Price", "Power", "Kilometers_Per_Year", "Is_First_Owner"]
].copy()

# Filter: only first-owner cars
analysis_goal = analysis_goal[analysis_goal["Is_First_Owner"] == 1]

# Rename
analysis_goal = analysis_goal.rename(columns={"Location": "City"})

# Group by + summarize
analysis_goal = analysis_goal.groupby("Brand").agg(
    Average_Price_Lakh=("Price", lambda x: round(x.mean(), 2)),
    Average_Power_BHP=("Power", lambda x: round(x.mean(), 2)),
    Average_Kilometers_Per_Year=("Kilometers_Per_Year", lambda x: round(x.mean(), 2)),
    Count=("Brand", "count")
).reset_index()

# Mutate
analysis_goal["Popular_list"] = analysis_goal["Count"].apply(
    lambda x: "Yes" if x > 50 else "No"
)

# Arrange
analysis_goal = analysis_goal.sort_values(
    by=["Popular_list", "Average_Price_Lakh"],
    ascending=[False, False]
)

# Save
save_dir = Path("../results")
save_dir.mkdir(parents=True, exist_ok=True)

save_path = save_dir / "analysis_goal.csv"
analysis_goal.to_csv(save_path, index=False)

print("✅ Dataframe successfully saved to:", save_path.resolve())

print("\n--- Final Analysis Report (Task e) ---")
display(analysis_goal.head(10))

--- Demonstrating pandas operations (Task e) ---
✅ Dataframe successfully saved to: /results/analysis_goal.csv

--- Final Analysis Report (Task e) ---


,Brand,Average_Price_Lakh,Average_Power_BHP,Average_Kilometers_Per_Year,Count,Popular_list
17,Mercedes-Benz,29.06,192.66,3444.53,260,Yes
1,BMW,27.56,211.24,4090.77,205,Yes
0,Audi,26.21,191.74,3947.15,188,Yes
25,Toyota,12.67,128.80,5648.36,320,Yes
15,Mahindra,8.53,121.79,4964.91,219,Yes
23,Skoda,8.21,128.58,4736.31,144,Yes
7,Ford,7.99,100.33,4304.88,232,Yes
9,Hyundai,5.83,92.36,3651.15,882,Yes
22,Renault,5.77,85.50,4127.91,127,Yes
8,Honda,5.73,106.12,3762.37,499,Yes


In [47]:
print(engineer_used_cars.columns)
print(encoded_used_cars.columns)

Index(['Unnamed: 0', 'Name', 'Location', 'Year', 'Kilometers_Driven',
       'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power',
       'Seats', 'New_Price', 'Price', 'Car_Age', 'Brand',
       'Kilometers_Per_Year', 'Is_First_Owner'],
      dtype='object')
Index(['Unnamed: 0', 'Name', 'Year', 'Kilometers_Driven', 'Owner_Type',
       'Mileage', 'Engine', 'Power', 'Seats', 'New_Price', 'Price', 'Car_Age',
       'Kilometers_Per_Year', 'Is_First_Owner', 'Fuel_Type_Diesel',
       'Fuel_Type_Electric', 'Fuel_Type_Petrol', 'Transmission_Automatic',
       'Transmission_Manual', 'Brand_Ambassador', 'Brand_Audi', 'Brand_BMW',
       'Brand_Bentley', 'Brand_Chevrolet', 'Brand_Datsun', 'Brand_Fiat',
       'Brand_Force', 'Brand_Ford', 'Brand_Honda', 'Brand_Hyundai',
       'Brand_ISUZU', 'Brand_Isuzu', 'Brand_Jaguar', 'Brand_Jeep',
       'Brand_Lamborghini', 'Brand_Land', 'Brand_Mahindra', 'Brand_Maruti',
       'Brand_Mercedes-Benz', 'Brand_Mini', 'Brand_Mitsubishi', 'B

# **Save data clean and results**

In [56]:
from pathlib import Path
import os

# Create folders
data_clean_dir = Path("data_clean")
results_dir = Path("results")

data_clean_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

# Save files
imputed_used_cars.to_csv(data_clean_dir / "used_cars_cleaned_imputed.csv", index=False)
encoded_used_cars.to_csv(data_clean_dir / "clean_used_cars_task_c.csv", index=False)
engineer_used_cars.to_csv(data_clean_dir / "used_cars_final_q1.csv", index=False)
analysis_goal.to_csv(results_dir / "analysis_goal.csv", index=False)

print("Saved files:")
for root, dirs, files in os.walk("."):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

Saved files:
./data_clean/used_cars_cleaned_imputed.csv
./data_clean/used_cars_final_q1.csv
./data_clean/clean_used_cars_task_c.csv
./results/analysis_goal.csv
./sample_data/california_housing_train.csv
./sample_data/mnist_test.csv
./sample_data/mnist_train_small.csv
./sample_data/california_housing_test.csv


In [57]:
import zipfile
from pathlib import Path

zip_path = Path("Q1_outputs.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ["data_clean", "results"]:
        for file_path in Path(folder).rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=file_path.as_posix())

print("Created zip file:", zip_path.resolve())

Created zip file: /content/Q1_outputs.zip


In [58]:
from google.colab import files
files.download("Q1_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>